# SimPandas Tutorial

SimPandas wraps `pandas.DataFrame` and `pandas.Series` with first-class **unit** support.
This notebook walks through every SimPandas-specific feature using two real data sources:

| Variable | Source |
|---|---|
| `sample_prod` | Excel file – two producers (`P1` field units, `P2` metric), weekly timesteps |
| `sample_smry` | Eclipse binary summary (`.SMSPEC` + `.UNSMRY`) |


## 1  Setup

In [1]:
import simpandas as spd

print('simpandas', spd.__version__)

simpandas 0.90.0


## 2  Loading data

Both readers return a `SimDataFrame` with unit metadata already attached.


In [2]:
sample_prod = spd.read_excel('./test/_testing_data/sample_prod.xlsx')
sample_prod

D:\git\simpandas\src\simpandas\readers\xlsx.py:80: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  excelread = pandas.read_excel(io, sheet_name=sheet_name, header=header, names=names, index_col=index_col,


,DATE,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,date,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
0,2100-01-01,4872.000000,0.000000,5012.100000,536.000000,0.000000,245.070000
1,2100-01-08,4838.015086,12.707564,4987.600782,533.380024,2.619792,245.018541
2,2100-01-15,4804.267236,25.304309,4963.221317,530.772855,5.226413,244.967092
3,2100-01-22,4770.754795,37.791219,4938.961019,528.178429,7.819928,244.915655
4,2100-01-29,4737.476123,50.169277,4914.819305,525.596686,10.400402,244.864228
...,...,...,...,...,...,...,...
361,2106-12-03,389.264791,6782.302094,854.665589,91.398966,433.506733,227.178007
362,2106-12-10,386.549452,7096.137397,850.487971,90.952206,433.911970,227.130305
363,2106-12-17,383.853055,7443.580545,846.330773,90.507630,434.315018,227.082613


In [3]:
sample_smry = spd.read_summary('./test/_testing_data/SAMPLE_PROD.SMSPEC')
sample_smry

,TIME,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,DAYS,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,,
1900-01-01,0.0,4872.000000,0.000000,5012.100098,536.000000,0.000000,245.070007
1900-01-02,1.0,4838.015137,12.707564,4987.600586,533.380005,2.619792,245.018539
1900-01-03,2.0,4804.267090,25.304308,4963.221191,530.772827,5.226413,244.967087
1900-01-04,3.0,4770.754883,37.791218,4938.960938,528.178406,7.819928,244.915649
1900-01-05,4.0,4737.476074,50.169277,4914.819336,525.596680,10.400402,244.864227
...,...,...,...,...,...,...,...
1900-12-28,361.0,389.264801,6782.302246,854.665588,91.398964,433.506744,227.178009
1900-12-29,362.0,386.549438,7096.137207,850.487976,90.952209,433.911957,227.130310


## 3  Units and metadata

Every `SimDataFrame` / `SimSeries` carries unit metadata alongside the data.


In [4]:
# Both objects are SimDataFrame, not bare pandas DataFrames
print(type(sample_prod))
print(type(sample_smry))

SimDataFrame
SimDataFrame


In [5]:
# .units returns a {column: unit_string} dict
sample_prod.units

ColumnUnits({'DATE': 'date', 'WOPR:P1': 'stb/day', 'WWPR:P1': 'stb/day', 'WBHP:P1': 'psia', 'WOPR:P2': 'sm3/day', 'WWPR:P2': 'sm3/day', 'WBHP:P2': 'barsa'})

In [6]:
# get_units() for the Eclipse summary
sample_smry.get_units()

{'TIME': 'DAYS',
 'WOPR:P1': 'stb/day',
 'WWPR:P1': 'stb/day',
 'WBHP:P1': 'psia',
 'WOPR:P2': 'sm3/day',
 'WWPR:P2': 'sm3/day',
 'WBHP:P2': 'barsa',
 'DATE': 'datetime'}

In [7]:
# .wells lists the unique well names from column names (right of ':')
print('sample_prod wells:', sample_prod.wells)
print('sample_smry wells:', sample_smry.wells)

sample_prod wells: ['P2', 'P1']
sample_smry wells: ['P2', 'P1']


In [8]:
# .meta carries reader-supplied metadata (e.g. DIMENS, STARTDAT from SMSPEC header)
sample_smry.meta

{'dimens': [1, 1, 1], 'startdat': [1, 1, 1900]}

## 4  Setting a DatetimeIndex

Most time-series helpers require a `DatetimeIndex`.
`set_index` with `inplace=True` works exactly like pandas —
the result is still a `SimDataFrame` with units intact.


In [9]:
sample_prod.set_index('DATE', inplace=True)
sample_prod

,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,
2100-01-01,4872.000000,0.000000,5012.100000,536.000000,0.000000,245.070000
2100-01-08,4838.015086,12.707564,4987.600782,533.380024,2.619792,245.018541
2100-01-15,4804.267236,25.304309,4963.221317,530.772855,5.226413,244.967092
2100-01-22,4770.754795,37.791219,4938.961019,528.178429,7.819928,244.915655
2100-01-29,4737.476123,50.169277,4914.819305,525.596686,10.400402,244.864228
...,...,...,...,...,...,...
2106-12-03,389.264791,6782.302094,854.665589,91.398966,433.506733,227.178007
2106-12-10,386.549452,7096.137397,850.487971,90.952206,433.911970,227.130305


In [10]:
# index.units is preserved after set_index
sample_prod.index.units

'date'

## 5  Smart column selection with `[]`

SimPandas's `__getitem__` goes beyond plain pandas:
it understands Eclipse-style `KEYWORD:WELLNAME` column names **and** can
select by date when the key looks like a date string.


In [11]:
# Exact column name -> SimSeries (with the column unit)
single_column = sample_prod['WOPR:P1']
print(type(single_column))
single_column

SimSeries


DATE
2100-01-01    4872.000000
2100-01-08    4838.015086
2100-01-15    4804.267236
2100-01-22    4770.754795
2100-01-29    4737.476123
                 ...     
2106-12-03     389.264791
2106-12-10     386.549452
2106-12-17     383.853055
2106-12-24     381.175466
2106-12-31     378.516555
Name: WOPR:P1, Length: 366, dtype: float64, units: stb/day

In [12]:
# Single name in a list -> 1-column SimDataFrame
sample_prod[['WOPR:P1']]

,WOPR:P1
,stb/day
DATE,
2100-01-01,4872.000000
2100-01-08,4838.015086
2100-01-15,4804.267236
2100-01-22,4770.754795
2100-01-29,4737.476123
...,...
2106-12-03,389.264791
2106-12-10,386.549452


In [13]:
# Multiple columns as a list
sample_prod[['WOPR:P1', 'WOPR:P2']]

,WOPR:P1,WOPR:P2
,stb/day,sm3/day
DATE,,
2100-01-01,4872.000000,536.000000
2100-01-08,4838.015086,533.380024
2100-01-15,4804.267236,530.772855
2100-01-22,4770.754795,528.178429
2100-01-29,4737.476123,525.596686
...,...,...
2106-12-03,389.264791,91.398966
2106-12-10,386.549452,90.952206


In [14]:
# Left part of 'KEYWORD:WELL' -> all columns with that keyword
sample_prod['WOPR']    # returns WOPR:P1 and WOPR:P2

,WOPR:P2,WOPR:P1
,sm3/day,stb/day
DATE,,
2100-01-01,536.000000,4872.000000
2100-01-08,533.380024,4838.015086
2100-01-15,530.772855,4804.267236
2100-01-22,528.178429,4770.754795
2100-01-29,525.596686,4737.476123
...,...,...
2106-12-03,91.398966,389.264791
2106-12-10,90.952206,386.549452


In [15]:
# Right part -> all measurements for that well
sample_prod['P1']    # returns WOPR:P1, WWPR:P1, WBHP:P1

,WOPR:P1,WWPR:P1,WBHP:P1
,stb/day,stb/day,psia
DATE,,,
2100-01-01,4872.000000,0.000000,5012.100000
2100-01-08,4838.015086,12.707564,4987.600782
2100-01-15,4804.267236,25.304309,4963.221317
2100-01-22,4770.754795,37.791219,4938.961019
2100-01-29,4737.476123,50.169277,4914.819305
...,...,...,...
2106-12-03,389.264791,6782.302094,854.665589
2106-12-10,386.549452,7096.137397,850.487971


In [16]:
# Slice of column names (inclusive on both ends)
sample_prod['WWPR:P1':'WOPR:P2']

,WWPR:P1,WBHP:P1,WOPR:P2
,stb/day,psia,sm3/day
DATE,,,
2100-01-01,0.000000,5012.100000,536.000000
2100-01-08,12.707564,4987.600782,533.380024
2100-01-15,25.304309,4963.221317,530.772855
2100-01-22,37.791219,4938.961019,528.178429
2100-01-29,50.169277,4914.819305,525.596686
...,...,...,...
2106-12-03,6782.302094,854.665589,91.398966
2106-12-10,7096.137397,850.487971,90.952206


In [17]:
# fnmatch-style wildcard pattern
sample_prod['W?PR*']    # any 4-char keyword starting with W

,WOPR:P1,WWPR:P1,WOPR:P2,WWPR:P2
,stb/day,stb/day,sm3/day,sm3/day
DATE,,,,
2100-01-01,4872.000000,0.000000,536.000000,0.000000
2100-01-08,4838.015086,12.707564,533.380024,2.619792
2100-01-15,4804.267236,25.304309,530.772855,5.226413
2100-01-22,4770.754795,37.791219,528.178429,7.819928
2100-01-29,4737.476123,50.169277,525.596686,10.400402
...,...,...,...,...
2106-12-03,389.264791,6782.302094,91.398966,433.506733
2106-12-10,386.549452,7096.137397,90.952206,433.911970


In [18]:
# When key is not a column, [] looks in the DatetimeIndex
# Exact date -> SimSeries (one row of all columns)
sample_prod['2100-04-09']

,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,
2100-04-09,4417.193459,168.173679,4679.798198,500.463246,35.501945,244.350552


In [19]:
# Partial date string -> all rows within that month
sample_prod['2100-04']

,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,
2100-04-02,4448.222288,156.831739,4702.785482,502.921534,33.048379,244.401871
2100-04-09,4417.193459,168.173679,4679.798198,500.463246,35.501945,244.350552
2100-04-16,4386.381074,179.417105,4656.923276,498.016974,37.943165,244.299244
2100-04-23,4355.783623,190.562918,4634.160167,495.582660,40.372098,244.247947
2100-04-30,4325.399605,201.612013,4611.508325,493.160245,42.788806,244.196660


In [20]:
# Year only -> all rows in that year
sample_prod['2100']

,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,
2100-01-01,4872.000000,0.000000,5012.100000,536.000000,0.000000,245.070000
2100-01-08,4838.015086,12.707564,4987.600782,533.380024,2.619792,245.018541
2100-01-15,4804.267236,25.304309,4963.221317,530.772855,5.226413,244.967092
2100-01-22,4770.754795,37.791219,4938.961019,528.178429,7.819928,244.915655
2100-01-29,4737.476123,50.169277,4914.819305,525.596686,10.400402,244.864228
2100-02-05,4704.429588,62.439454,4890.795597,523.027561,12.967899,244.812812
2100-02-12,4671.613571,74.602718,4866.889316,520.470995,15.522484,244.761406
2100-02-19,4639.026464,86.660027,4843.099890,517.926925,18.064221,244.710012


In [21]:
# Date range slice
sample_prod['2100-04-09':'2100-05-15']

,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,
2100-04-09,4417.193459,168.173679,4679.798198,500.463246,35.501945,244.350552
2100-04-16,4386.381074,179.417105,4656.923276,498.016974,37.943165,244.299244
2100-04-23,4355.783623,190.562918,4634.160167,495.582660,40.372098,244.247947
2100-04-30,4325.399605,201.612013,4611.508325,493.160245,42.788806,244.196660
2100-05-07,4295.227534,212.565279,4588.967205,490.749670,45.193350,244.145384
2100-05-14,4265.265929,223.423599,4566.536266,488.350879,47.585790,244.094119


### `[index, column]` shortcut

Combine a date/date-range with a column name or name-pattern in one `[]` call using a 2-tuple.


In [22]:
sample_prod['2100-02-26', 'WBHP:P1']

,WBHP:P1
,psia
DATE,
2100-01-01,5012.100000
2100-01-08,4987.600782
2100-01-15,4963.221317
2100-01-22,4938.961019
2100-01-29,4914.819305
...,...
2106-12-03,854.665589
2106-12-10,850.487971


In [23]:
# Date range + right-part well filter
sample_prod['2100-04':'2100-06', 'P1']

,WOPR:P1,WWPR:P1,WBHP:P1
,stb/day,stb/day,psia
DATE,,,
2100-04-02,4448.222288,156.831739,4702.785482
2100-04-09,4417.193459,168.173679,4679.798198
2100-04-16,4386.381074,179.417105,4656.923276
2100-04-23,4355.783623,190.562918,4634.160167
2100-04-30,4325.399605,201.612013,4611.508325
2100-05-07,4295.227534,212.565279,4588.967205
2100-05-14,4265.265929,223.423599,4566.536266
2100-05-21,4235.513323,234.187850,4544.214971


### `SimSeries[]` understands date strings too

In [24]:
single_column['2100-05']    # all values in May 2100

DATE
2100-05-07    4295.227534
2100-05-14    4265.265929
2100-05-21    4235.513323
2100-05-28    4205.968258
Name: WOPR:P1, dtype: float64, units: stb/day

In [25]:
single_column['2100-04-30':'2100-06-12']

DATE
2100-04-30    4325.399605
2100-05-07    4295.227534
2100-05-14    4265.265929
2100-05-21    4235.513323
2100-05-28    4205.968258
2100-06-04    4176.629286
2100-06-11    4147.494970
Name: WOPR:P1, dtype: float64, units: stb/day

## 6  `.loc` and `.iloc`

SimPandas wraps both indexers so results always come back as `SimDataFrame` / `SimSeries`
with unit metadata preserved.  All pandas-compatible date-string shortcuts work.


In [26]:
sample_prod.loc['2100-02-26']        # one row by exact date -> SimSeries

,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,
2100-02-26,4606.66667,98.612335,4819.426748,515.395291,20.593174,244.658628


In [27]:
sample_prod.loc['2100-01']           # all rows in January 2100

,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,
2100-01-01,4872.0,0.0,5012.1,536.0,0.0,245.07


In [28]:
sample_prod.loc['2100']              # full year 2100

,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,
2100-01-01,4872.0,0.0,5012.1,536.0,0.0,245.07


In [29]:
# Date range + right-part column filter
sample_prod.loc['2100-12-24':'2101-01-15', 'P1']

,WOPR:P1,WWPR:P1,WBHP:P1
,stb/day,stb/day,psia
DATE,,,
2100-12-24,3409.291609,525.744549,3903.817767
2100-12-31,3385.509901,533.928441,3884.735849
2101-01-07,3361.894083,542.044180,3865.747203
2101-01-14,3338.442999,550.092469,3846.851375


In [30]:
# iloc still returns SimDataFrame / SimSeries
print(type(sample_prod.iloc[0:5]))
sample_prod.iloc[0:5, 0:3]

SimDataFrame


,WOPR:P1,WWPR:P1,WBHP:P1
,stb/day,stb/day,psia
DATE,,,
2100-01-01,4872.000000,0.000000,5012.100000
2100-01-08,4838.015086,12.707564,4987.600782
2100-01-15,4804.267236,25.304309,4963.221317
2100-01-22,4770.754795,37.791219,4938.961019
2100-01-29,4737.476123,50.169277,4914.819305


## 7  Time aggregations

`.daily()`, `.monthly()`, and `.yearly()` resample a `DatetimeIndex` frame
and return a new `SimDataFrame` with units intact.


In [31]:
# Mean per calendar day, DatetimeIndex (default)
sample_prod.daily()

,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,
2100-01-01,4872.000000,0.000000,5012.100000,536.000000,0.000000,245.070000
2100-01-08,4838.015086,12.707564,4987.600782,533.380024,2.619792,245.018541
2100-01-15,4804.267236,25.304309,4963.221317,530.772855,5.226413,244.967092
2100-01-22,4770.754795,37.791219,4938.961019,528.178429,7.819928,244.915655
2100-01-29,4737.476123,50.169277,4914.819305,525.596686,10.400402,244.864228
...,...,...,...,...,...,...
2106-12-03,389.264791,6782.302094,854.665589,91.398966,433.506733,227.178007
2106-12-10,386.549452,7096.137397,850.487971,90.952206,433.911970,227.130305


In [32]:
# Integer index instead of DatetimeIndex
sample_prod.daily(datetime_index=False)

WOPR:P1      WWPR:P1      WBHP:P1     WOPR:P2     WWPR:P2  \
                    stb/day      stb/day         psia     sm3/day     sm3/day   
YEAR MONTH DAY                                                                  
2100 1     1    4872.000000     0.000000  5012.100000  536.000000    0.000000   
           8    4838.015086    12.707564  4987.600782  533.380024    2.619792   
           15   4804.267236    25.304309  4963.221317  530.772855    5.226413   
           22   4770.754795    37.791219  4938.961019  528.178429    7.819928   
           29   4737.476123    50.169277  4914.819305  525.596686   10.400402   
...                     ...          ...          ...         ...         ...   
2106 12    3     389.264791  6782.302094   854.665589   91.398966  433.506733   
           10    386.549452  7096.137397   850.487971   90.952206  433.911970   
           17    383.853055  7443.580545   846.330773   90.507630  434.315018   
           24    381.175466  7830.322069   842.193896   90.065228  434.715890   
           31    378.516555  8263.413916   838.077240   89.624988  435.114595   

                   WBHP:P2  
                     barsa  
YEAR MONTH DAY              
2100 1     1    245.070000  
           8    245.018541  
           15   244.967092  
           22   244.915655  
           29   244.864228  
...                    ...  
2106 12    3    227.178007  
           10   227.130305  
           17   227.082613  
           24   227.034930  
           31   226.987258  

[366 rows x 6 columns]

In [33]:
# complete_index=True fills gaps with NaN rows for missing days
sample_prod.daily(complete_index=True)

D:\git\simpandas\src\simpandas\basics.py:1569: FutureWarning: The 'downcast' keyword in DataFrame.interpolate is deprecated and will be removed in a future version. Call result.infer_objects(copy=False) on the result instead.
  return self._class(data=self.as_pandas().interpolate(method=method, axis=axis, limit=limit, inplace=inplace,


,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,
2100-01-01,4872.000000,0.000000,5012.100000,536.000000,0.000000,245.070000
2100-01-02,4867.145012,1.815366,5008.600112,535.625718,0.374256,245.062649
2100-01-03,4862.290025,3.630733,5005.100223,535.251435,0.748512,245.055297
2100-01-04,4857.435037,5.446099,5001.600335,534.877153,1.122768,245.047946
2100-01-05,4852.580049,7.261465,4998.100447,534.502871,1.497024,245.040595
...,...,...,...,...,...,...
2106-12-27,380.035932,8015.932861,840.429615,89.876553,434.886763,227.014499
2106-12-28,379.656088,8077.803125,839.841521,89.813662,434.943721,227.007689


In [34]:
sample_prod.monthly()

WOPR:P1      WWPR:P1      WBHP:P1     WOPR:P2     WWPR:P2  \
                stb/day      stb/day         psia     sm3/day     sm3/day   
YEAR MONTH                                                                  
2100 1      4804.502648    25.194474  4963.340484  530.785599    5.213307   
     2      4655.434073    80.578634  4855.052888  519.205193   16.786945   
     3      4526.889935   127.976341  4760.820347  509.127852   26.852186   
     4      4386.596010   179.319491  4657.035090  498.028932   37.930879   
     5      4250.493761   228.758908  4555.430306  487.163194   48.769981   
...                 ...          ...          ...         ...         ...   
2106 8       433.889735  4142.180356   922.119914   98.612612  426.936026   
     9       421.909330  4581.346510   904.222333   96.698623  428.684233   
     10      408.833838  5233.033167   884.510406   94.590606  430.605619   
     11      396.149012  6129.131034   865.212615   92.526877  432.482607   
     12      383.871864  7483.151204   846.351094   90.509804  434.312841   

               WBHP:P2  
                 barsa  
YEAR MONTH              
2100 1      244.967103  
     2      244.735714  
     3      244.530223  
     4      244.299255  
     5      244.068497  
...                ...  
2106 8      227.918683  
     9      227.727312  
     10     227.512215  
     11     227.297313  
     12     227.082623  

[84 rows x 6 columns]

In [35]:
# complete_index fills months without data; ffill propagates last known value
sample_prod.monthly(complete_index=True, fillna_method='ffill')

D:\git\simpandas\src\simpandas\basics.py:1605: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data=self.as_pandas().fillna(value=value, method=method, axis=axis, inplace=inplace, limit=limit,
D:\git\simpandas\src\simpandas\basics.py:1605: FutureWarning: The 'downcast' keyword in fillna is deprecated and will be removed in a future version. Use res.infer_objects(copy=False) to infer non-object dtype, or pd.to_numeric with the 'downcast' keyword to downcast numeric results.
  data=self.as_pandas().fillna(value=value, method=method, axis=axis, inplace=inplace, limit=limit,


WOPR:P1      WWPR:P1      WBHP:P1     WOPR:P2     WWPR:P2  \
                stb/day      stb/day         psia     sm3/day     sm3/day   
YEAR MONTH                                                                  
2100 1      4813.151232    21.971919  4969.601282  531.455136    4.544005   
     2      4674.121138    73.658197  4868.680396  520.662535   15.330834   
     3      4538.713412   123.625694  4769.509388  510.057068   25.924242   
     4      4401.931060   173.730016  4668.435462  499.248101   36.714344   
     5      4269.300586   221.946606  4569.519038  488.669860   47.267371   
...                 ...          ...          ...         ...         ...   
2106 8       435.416257  4095.353491   924.387337   98.855093  426.714258   
     9       422.405014  4561.993971   904.964407   96.777982  428.611783   
     10      409.569779  5188.259952   885.626134   94.709924  430.497008   
     11      397.356503  6031.363569   867.055993   92.724010  432.303460   
     12      385.432720  7268.537834   848.759751   90.767388  434.079370   

               WBHP:P2  
                 barsa  
YEAR MONTH              
2100 1      244.980377  
     2      244.765086  
     3      244.549276  
     4      244.324905  
     5      244.100742  
...                ...  
2106 8      227.942618  
     9      227.735284  
     10     227.524543  
     11     227.318001  
     12     227.110313  

[84 rows x 6 columns]

In [36]:
# Report on the first day of each month
sample_prod.monthly(day='first')

,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,
2100-01-01,4804.502648,25.194474,4963.340484,530.785599,5.213307,244.967103
2100-02-05,4655.434073,80.578634,4855.052888,519.205193,16.786945,244.735714
2100-03-05,4526.889935,127.976341,4760.820347,509.127852,26.852186,244.530223
2100-04-02,4386.596010,179.319491,4657.035090,498.028932,37.930879,244.299255
2100-05-07,4250.493761,228.758908,4555.430306,487.163194,48.769981,244.068497
...,...,...,...,...,...,...
2106-08-06,433.889735,4142.180356,922.119914,98.612612,426.936026,227.918683
2106-09-03,421.909330,4581.346510,904.222333,96.698623,428.684233,227.727312


In [37]:
sample_prod.yearly()

,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
YEAR,,,,,,
2100,4084.629255,286.516370,4424.965940,473.211178,62.636817,243.736822
2101,2827.851038,722.344020,3420.913611,365.836615,169.202915,241.064336
2102,1965.048786,1012.729855,2651.452633,283.549532,250.133287,238.446234
2103,1365.495099,1245.090190,2055.065361,219.771160,312.198137,235.856566
2104,948.870521,1510.798159,1592.822585,170.338362,359.695564,233.295024
2105,659.361770,2007.256589,1234.551385,132.024409,395.944791,230.761302
2106,456.681516,4112.786245,954.624300,102.088670,423.728074,228.231176


## 8  Unit-aware arithmetic

Operators `+`, `-`, `*`, `/`, `**` all track units.
When two operands have **convertible** units (e.g. `stb/day` <-> `sm3/day`),
SimPandas converts the right-hand side to the left-hand side's unit automatically.


In [38]:
# P1 columns: rates in stb/day (field units)
# P2 columns: rates in sm3/day and barsa (metric units)
print('P1 units:', sample_prod['P1'].units)
print('P2 units:', sample_prod['P2'].units)

P1 units: ColumnUnits({'WOPR:P1': 'stb/day', 'WWPR:P1': 'stb/day', 'WBHP:P1': 'psia'})
P2 units: ColumnUnits({'WOPR:P2': 'sm3/day', 'WWPR:P2': 'sm3/day', 'WBHP:P2': 'barsa'})


In [39]:
# sum(axis=1): sum across columns (wells) for each timestep
# P2 auto-converted to stb/day to match P1 (left operand)
total_oil_rate = sample_prod['WOPR'].sum(axis=1)
print('result unit:', total_oil_rate.get_units())
total_oil_rate

result unit: {'WOPR:P.sum': 'sm3/day', 'DATE': 'date'}


DATE
2100-01-01    1310.585703
2100-01-08    1302.562561
2100-01-15    1294.589915
2100-01-22    1286.667440
2100-01-29    1278.794812
                 ...     
2106-12-03     153.287090
2106-12-10     152.408626
2106-12-17     151.535358
2106-12-24     150.667253
2106-12-31     149.804280
Name: WOPR:P.sum, Length: 366, dtype: float64, units: sm3/day

In [40]:
# Direct addition: P1 (stb/day) + P2 (sm3/day)
p1_plus_p2 = sample_prod['WOPR:P1'] + sample_prod['WOPR:P2']
print('result unit:', p1_plus_p2.get_units())
p1_plus_p2

result unit: {'WOPR:P1+P2': 'stb/day', 'DATE': 'date'}


DATE
2100-01-01    8243.340304
2100-01-08    8192.876229
2100-01-15    8142.729769
2100-01-22    8092.898875
2100-01-29    8043.381514
                 ...     
2106-12-03     964.147285
2106-12-10     958.621912
2106-12-17     953.129216
2106-12-24     947.668997
2106-12-31     942.241057
Name: WOPR:P1+P2, Length: 366, dtype: float64, units: stb/day

In [41]:
# Reversed operands: result unit is now sm3/day
p2_plus_p1 = sample_prod['WOPR:P2'] + sample_prod['WOPR:P1']
print('result unit:', p2_plus_p1.get_units())
p2_plus_p1

result unit: {'WOPR:P2+P1': 'sm3/day', 'DATE': 'date'}


DATE
2100-01-01    1310.585703
2100-01-08    1302.562561
2100-01-15    1294.589915
2100-01-22    1286.667440
2100-01-29    1278.794812
                 ...     
2106-12-03     153.287090
2106-12-10     152.408626
2106-12-17     151.535358
2106-12-24     150.667253
2106-12-31     149.804280
Name: WOPR:P2+P1, Length: 366, dtype: float64, units: sm3/day

## 9  Aggregation methods

All scalar/per-column aggregation results carry the correct unit.
`rms()` (root-mean-square) is a SimPandas extension not in standard pandas.


In [42]:
s = sample_prod['WOPR:P1']   # SimSeries - unit = stb/day

print('sum  :', s.sum())
print('mean :', s.mean())
print('std  :', s.std())
print('var  :', s.var())
print('rms  :', s.rms())    # root-mean-square -- SimPandas extension
print('min  :', s.min())
print('max  :', s.max())

sum  : 644554.0859550093_stb/day
mean : 1761.0767375819928_stb/day
std  : 1240.9455461402374_stb/day
var  : 1539945.8484852922_(stb/day)2
rms  : 2153.399549971965_stb/day
min  : 378.51655463189707_stb/day
max  : 4872.0_stb/day


In [43]:
# describe() keeps unit labels in the output
sample_prod.describe()

,WOPR:P1,WWPR:P1,WBHP:P1,WOPR:P2,WWPR:P2,WBHP:P2
,stb/day,stb/day,psia,sm3/day,sm3/day,barsa
DATE,,,,,,
count,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000
mean,1761.076738,1560.301686,2335.432166,249.753924,281.722468,235.913453
std,1240.945546,1309.777718,1180.190538,126.210995,122.946358,5.241234
min,378.516555,0.000000,838.077240,89.624988,0.000000,226.987258
25%,716.955955,808.038650,1310.596323,140.156746,192.016091,231.378845
50%,1357.996776,1244.021621,2049.524859,219.178653,312.799679,235.855396
75%,2572.193711,1817.810680,3205.064574,342.753459,388.330114,240.418556
max,4872.000000,8263.413916,5012.100000,536.000000,435.114595,245.070000


## 10  Unit conversion with `.to()`

Convert any `SimDataFrame` or `SimSeries` to a target unit in one call.


In [44]:
# SimSeries: stb/day -> sm3/day
p1_metric = sample_prod['WOPR:P1'].to('sm3/day')
print('original :', sample_prod['WOPR:P1'].get_units())
print('converted:', p1_metric.get_units())
p1_metric

original : {'WOPR:P1': 'stb/day', 'DATE': 'date'}
converted: {'WOPR:P1': 'sm3/day', 'DATE': 'date'}


DATE
2100-01-01    774.585703
2100-01-08    769.182536
2100-01-15    763.817060
2100-01-22    758.489010
2100-01-29    753.198127
                 ...    
2106-12-03     61.888124
2106-12-10     61.456420
2106-12-17     61.027727
2106-12-24     60.602025
2106-12-31     60.179292
Name: WOPR:P1, Length: 366, dtype: float64, units: sm3/day

In [45]:
# SimDataFrame: convert every convertible column at once
sample_prod[['WBHP:P1', 'WBHP:P2']].to('barsa')

,WBHP:P1,WBHP:P2
,barsa,barsa
DATE,,
2100-01-01,345.572130,245.070000
2100-01-08,343.882969,245.018541
2100-01-15,342.202064,244.967092
2100-01-22,340.529375,244.915655
2100-01-29,338.864862,244.864228
...,...,...
2106-12-03,58.927118,227.178007
2106-12-10,58.639081,227.130305


## 11  Equality comparisons with precision

`.eq(other, precision=N)` rounds each value to `N` significant figures before comparing.
Useful when two results share the same physical meaning but were computed via
different unit conversion paths.


In [46]:
p1_p2 = sample_prod['WOPR:P1'] + sample_prod['WOPR:P2']   # result in stb/day
p2_p1 = sample_prod['WOPR:P2'] + sample_prod['WOPR:P1']   # result in sm3/day

# Exact == may fail due to floating-point round-trip
print('exact equal     :', (p1_p2 == p2_p1).all())
# .eq() rounds to N significant figures before comparing
print('equal at 6 s.f. :', p1_p2.eq(p2_p1, precision=6).all())

exact equal     : False
equal at 6 s.f. : True


## 12  `SimSeries` with per-index units

A `SimSeries` can hold a **different unit for each element** by passing a `{label: unit}` dict.
This is ideal for parameter tables where each row has a distinct physical quantity.


In [47]:
params = spd.SimSeries(
    data=[1000, 0.25, 300, 365],
    index=['depth', 'porosity', 'pressure', 'time'],
    units={'depth': 'm', 'porosity': 'fraction', 'pressure': 'barsa', 'time': 'day'},
    name='reservoir_params'
)
print('units dict:', params.units)
params

units dict: {'depth': 'm', 'porosity': 'fraction', 'pressure': 'barsa', 'time': 'day'}



depth       1000.00    m
porosity       0.25    fraction
pressure     300.00    barsa
time         365.00    day
Name: reservoir_params, dtype: float64

In [48]:
# Retrieve the unit for a specific index label
print('depth unit   :', params.get_units('depth'))
print('pressure unit:', params.get_units('pressure'))

depth unit   : {'depth': 'm', 'porosity': 'fraction', 'pressure': 'barsa', 'time': 'day'}
pressure unit: {'depth': 'm', 'porosity': 'fraction', 'pressure': 'barsa', 'time': 'day'}


## 13  Unit-aware column assignment

Assigning a `(value, unit)` tuple to a column tells SimPandas the value's unit.
If the column already exists, the value is converted to the column's existing unit.
If the column is new, the unit is registered automatically.


In [49]:
test = sample_prod[['WOPR:P1', 'WWPR:P1']].copy()

# Assign a new column with an explicit unit
test['WOPR:P3'] = 500, 'sm3/day'
print('WOPR:P3 unit:', test.get_units('WOPR:P3'))
test.head(3)

WOPR:P3 unit: sm3/day


,WOPR:P1,WWPR:P1,WOPR:P3
,stb/day,stb/day,sm3/day
DATE,,,
2100-01-01,4872.000000,0.000000,500
2100-01-08,4838.015086,12.707564,500
2100-01-15,4804.267236,25.304309,500


In [50]:
# Overwrite an existing column -- value auto-converted to existing column unit
test['WOPR:P1'] = 1000, 'sm3/day'   # stored as stb/day equivalent
print('WOPR:P1 unit still:', test.get_units('WOPR:P1'))
test['WOPR:P1'].head(3)

WOPR:P1 unit still: sm3/day


DATE
2100-01-01    1000
2100-01-08    1000
2100-01-15    1000
Name: WOPR:P1, dtype: int64, units: sm3/day

## 14  Export

All standard pandas writers are available on `SimDataFrame` / `SimSeries`,
with SimPandas wrappers that embed unit metadata in the output files.


In [51]:
print('Writer methods on SimDataFrame:')
print([m for m in dir(sample_prod) if m.startswith('to_')])

Writer methods on SimDataFrame:
['to_DataFrame', 'to_DataFrameMultiIndex', 'to_Pandas', 'to_Series', 'to_SimDataFrame', 'to_SimSeries', 'to_SimationResults', 'to_clipboard', 'to_csv', 'to_dataframe', 'to_dict', 'to_excel', 'to_feather', 'to_frame', 'to_gbq', 'to_hdf', 'to_hdf5', 'to_html', 'to_json', 'to_latex', 'to_markdown', 'to_numpy', 'to_orc', 'to_pandas', 'to_parquet', 'to_period', 'to_pickle', 'to_prodml', 'to_records', 'to_resqml', 'to_series', 'to_simdataframe', 'to_simseries', 'to_sql', 'to_stata', 'to_string', 'to_summary', 'to_timestamp', 'to_witsml', 'to_xarray', 'to_xml']


In [52]:
# Uncomment any line below to actually write

# Eclipse binary summary (round-trip)
# sample_prod.to_summary('./output/ROUND_TRIP.SMSPEC')
# reloaded = spd.read_summary('./output/ROUND_TRIP.SMSPEC')

# CSV with units row
# sample_prod.to_csv('./output/sample_prod.csv')        # row 0 contains units
# df_back = spd.read_csv('./output/sample_prod.csv', units=0)

# JSON with units envelope
# sample_prod.to_json('./output/sample_prod.json')

# HDF5
# sample_prod.to_hdf5('./output/sample_prod.h5')

# Excel
# sample_prod.to_excel('./output/sample_prod.xlsx')

print('All writers preserve unit metadata for round-tripping.')

All writers preserve unit metadata for round-tripping.
